#### **Advanced Types, `Annotated` and Custom Types**

1. **Advanced Types** — `Literal`, `Union`, `Optional`, `Enum`, `List`, `Dict`, `Set`, `Tuple`, etc.
2. **`Annotated`** — attach Pydantic validation rules directly to Python type hints.
3. **Custom Types** — create reusable types with your own validation logic.

---
#### **Advanced Types**

Pydantic works with Python's type system, so you can create very precise data models.

**`Optional`:** `Optional[T]` means a value can be `T` or `None`.

In [3]:
from pydantic import BaseModel
from typing import Optional

class User(BaseModel):
    name: str
    age: Optional[int] = None

user = User(name="Shourov Roy")
print(user)

user1 = User(name="Shourov Roy", age=23)
print(user1)

name='Shourov Roy' age=None
name='Shourov Roy' age=23


**Important:** In modern Python, you can usually write:

```python
age: int | None = None
```

instead of:

```python
age: Optional[int] = None
```

---
**`Union`:** `Union` means a value can have multiple possible types.

In [6]:
from pydantic import BaseModel
from typing import Union

class Product(BaseModel):
    product_id: Union[int, str]

print(Product(product_id=101))
print(Product(product_id="ABC101"))

product_id=101
product_id='ABC101'


**Modern Python syntax:**

```python
class Product(BaseModel):
    product_id: int | str
```

---
**`Literal`:** `Literal` restricts a value to a specific set of allowed values.

In [7]:
from typing import Literal
from pydantic import BaseModel

class User(BaseModel):
    role: Literal["admin", "user", "guest"]

print(User(role="admin"))
print(User(role="user"))
# This is invalid 
# print(User(role="manager"))

role='admin'
role='user'


Pydantic raises a validation error because `"manager"` isn't one of the permitted values.

---
**`Enum`:** For larger sets of predefined values, `Enum` is often cleaner.

In [8]:
from enum import Enum
from pydantic import BaseModel

class UserRole(str, Enum):
    ADMIN = "admin"
    USER = "user"
    GUEST = "guest"


class User(BaseModel):
    name: str
    role: UserRole

user = User(name="John", role="admin")
print(user.role)
print(user.role.value)

UserRole.ADMIN
admin


---
**Lists:** Pydantic can validate every element inside a list.


In [9]:
from pydantic import BaseModel

class Student(BaseModel):
    name: str
    scores: list[int]

student = Student(name="John", scores=[80, 90, 95])
print(student)

name='John' scores=[80, 90, 95]


---
**Dictionaries:** You can specify both the key and value types.

In [10]:
class Product(BaseModel):
    prices: dict[str, float]

product = Product(prices={
        "USD": 20.5,
        "EUR": 18.2
    })
print(product)

prices={'USD': 20.5, 'EUR': 18.2}


---
**Sets:** Use `set` when duplicate values should be removed.

In [11]:
class User(BaseModel):
    tags: set[str]

user = User(tags=["python", "fastapi", "python"])
print(user)

tags={'fastapi', 'python'}


---
**Tuples:** Tuples can describe fixed structures.

```python
class Point(BaseModel):
    coordinates: tuple[float, float]
```

Valid:

```python
Point(coordinates=(10.5, 20.3))
```

Here `tuple[float, float]` means exactly two values, both expected to be floats. You can also define variable-length tuples:

```python
values: tuple[int, ...]
```

---

#### **`Annotated`**

Now we get to one of the most important advanced Pydantic features. `Annotated` comes from Python's `typing` module. It allows you to attach additional metadata or validation rules to a type. Basic example:

```python
from typing import Annotated
from pydantic import BaseModel, Field

class User(BaseModel):
    username: Annotated[str, Field(min_length=3, max_length=20)]
```

The underlying type is still `str` but Pydantic additionally knows:

```text
minimum length = 3
maximum length = 20
```
---

#### **Custom Types**

Sometimes built-in Python types aren't enough. For example, suppose your application needs a special type `PositiveInteger`. You could create reusable validation around it. One simple approach is using `Annotated`.

In [14]:
from typing import Annotated
from pydantic import Field
from pydantic import BaseModel

PositiveInt = Annotated[int, Field(gt=0)]
class Product(BaseModel):
    quantity: PositiveInt

print(Product(quantity=10))
# print(Product(quantity=-5)) this give error

quantity=10


---

#### **Custom Types with Validators**

For more complex requirements, you can define your own type and validation logic. For example, suppose you want a special type for a username.

In [15]:
from typing import Annotated
from pydantic import BaseModel, AfterValidator

def validate_username(value: str) -> str:
    if not value.isalnum():
        raise ValueError("Username must contain only letters and numbers")
    return value

Username = Annotated[str, AfterValidator(validate_username)]
class User(BaseModel):
    username: Username

print(User(username="john123"))
# print(User(username="john@123")) this is invalid

username='john123'
